In [2]:
import sys
import random
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn.functional as F
from torchvision.transforms.functional import gaussian_blur

# ==========================================================
# 1. PATHS
# ==========================================================

# FaithSR architecture
sys.path.append(
    "/kaggle/input/models/muhammadhassan2/faithsrnet-v01/other/default/1"
)

from faithsrnet_v01 import FaithSRNetV01

DATA_ROOT = Path(
    "/kaggle/input/datasets/nguyenquangnhat2100/"
    "ucmerced-landuse-dataset/UCMerced_LandUse"
)

# Automatically find uploaded checkpoints
from pathlib import Path

BASE_CKPT = Path(
    "/kaggle/input/models/muhammadhassan2/base-best/other/default/1/base_best.pth"
)

DA_CKPT = Path(
    "/kaggle/input/models/muhammadhassan2/da-best/other/default/1/da_best.pth"
)

print("BASE:", BASE_CKPT)
print("DA:", DA_CKPT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ==========================================================
# 2. HELPERS
# ==========================================================

EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}

def pil_to_tensor(img):
    arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

def tensor_to_pil(x):
    arr = (
        x.detach()
        .clamp(0, 1)
        .permute(1, 2, 0)
        .cpu()
        .numpy() * 255.0 + 0.5
    ).astype(np.uint8)

    return Image.fromarray(arr)

def center_crop_224(img):
    w, h = img.size
    size = 224

    left = (w - size) // 2
    top = (h - size) // 2

    return img.crop((left, top, left + size, top + size))

def jpeg_roundtrip(x, quality):
    buf = BytesIO()

    tensor_to_pil(x).save(
        buf,
        format="JPEG",
        quality=quality,
        subsampling=0
    )

    buf.seek(0)

    return pil_to_tensor(
        Image.open(buf).convert("RGB")
    )

def psnr(x, y):
    mse = F.mse_loss(x, y)
    return 10.0 * torch.log10(1.0 / (mse + 1e-12))

def gaussian_window(
    window_size=11,
    sigma=1.5,
    channels=3,
    device="cpu"
):
    coords = torch.arange(
        window_size,
        device=device,
        dtype=torch.float32
    )

    coords -= window_size // 2

    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g /= g.sum()

    w2 = torch.outer(g, g)

    return w2.expand(
        channels, 1, window_size, window_size
    ).contiguous()

def ssim(x, y, window_size=11):
    c = x.shape[1]

    window = gaussian_window(
        window_size,
        1.5,
        c,
        x.device
    )

    pad = window_size // 2

    mu_x = F.conv2d(x, window, padding=pad, groups=c)
    mu_y = F.conv2d(y, window, padding=pad, groups=c)

    sigma_x = (
        F.conv2d(x * x, window, padding=pad, groups=c)
        - mu_x * mu_x
    )

    sigma_y = (
        F.conv2d(y * y, window, padding=pad, groups=c)
        - mu_y * mu_y
    )

    sigma_xy = (
        F.conv2d(x * y, window, padding=pad, groups=c)
        - mu_x * mu_y
    )

    c1 = 0.01 ** 2
    c2 = 0.03 ** 2

    score = (
        (2 * mu_x * mu_y + c1)
        * (2 * sigma_xy + c2)
    ) / (
        (mu_x ** 2 + mu_y ** 2 + c1)
        * (sigma_x + sigma_y + c2)
        + 1e-12
    )

    return score.mean()


# ==========================================================
# 3. RECREATE EXACT VALIDATION SPLIT
#    Same seed=42 and 80/20 split used during training
# ==========================================================

files = sorted([
    p for p in DATA_ROOT.rglob("*")
    if p.suffix.lower() in EXTS
])

rr = random.Random(42)
rr.shuffle(files)

split = int(len(files) * 0.8)
val_files = files[split:]

print("Validation images:", len(val_files))


# ==========================================================
# 4. LOAD MODELS
# ==========================================================

base_model = FaithSRNetV01(
    scale=4,
    use_degradation_adaptation=False,
    use_observation_consistency=False
).to(device)

da_model = FaithSRNetV01(
    scale=4,
    use_degradation_adaptation=True,
    use_observation_consistency=False
).to(device)

base_ckpt = torch.load(BASE_CKPT, map_location=device)
da_ckpt   = torch.load(DA_CKPT, map_location=device)

base_model.load_state_dict(base_ckpt["model"])
da_model.load_state_dict(da_ckpt["model"])

base_model.eval()
da_model.eval()

print(
    "BASE checkpoint epoch:",
    base_ckpt["epoch"]
)

print(
    "DA checkpoint epoch:",
    da_ckpt["epoch"]
)


# ==========================================================
# 5. FIXED DEGRADATION LEVELS
# ==========================================================

LEVELS = {
    "Bicubic": {
        "sigma": 0.0,
        "noise": 0.0,
        "jpeg": 100
    },

    "Mild": {
        "sigma": 0.6,
        "noise": 0.005,
        "jpeg": 90
    },

    "Medium": {
        "sigma": 1.2,
        "noise": 0.015,
        "jpeg": 75
    },

    "Severe": {
        "sigma": 2.0,
        "noise": 0.030,
        "jpeg": 60
    }
}


def make_test_lr(hr, level, seed):
    cfg = LEVELS[level]

    x = hr.clone()

    # Blur
    if cfg["sigma"] > 0:
        sigma = cfg["sigma"]

        kernel = (
            3 if sigma < 0.7
            else 5 if sigma < 1.5
            else 7
        )

        x = gaussian_blur(
            x,
            [kernel, kernel],
            [sigma, sigma]
        )

    # x4 downsample
    x = F.interpolate(
        x.unsqueeze(0),
        scale_factor=0.25,
        mode="bicubic",
        align_corners=False,
        antialias=True
    )[0]

    # Noise
    if cfg["noise"] > 0:
        g = torch.Generator()
        g.manual_seed(seed)

        noise = torch.randn(
            x.shape,
            generator=g,
            dtype=x.dtype
        ) * cfg["noise"]

        x = (x + noise).clamp(0, 1)

    # JPEG compression
    if cfg["jpeg"] < 100:
        x = jpeg_roundtrip(
            x,
            cfg["jpeg"]
        )

    return x.clamp(0, 1)


# ==========================================================
# 6. STRESS TEST
# ==========================================================

results = []

for level in LEVELS:

    bic_psnr_list = []
    bic_ssim_list = []

    base_psnr_list = []
    base_ssim_list = []

    da_psnr_list = []
    da_ssim_list = []

    print(f"\nTesting: {level}")

    for idx, path in enumerate(val_files):

        img = Image.open(path).convert("RGB")
        img = center_crop_224(img)

        hr = pil_to_tensor(img)

        lr = make_test_lr(
            hr,
            level,
            seed=1000 + idx
        )

        hr_b = hr.unsqueeze(0).to(device)
        lr_b = lr.unsqueeze(0).to(device)

        # Bicubic baseline
        bic = F.interpolate(
            lr_b,
            size=hr_b.shape[-2:],
            mode="bicubic",
            align_corners=False,
            antialias=True
        )

        with torch.no_grad():

            base_sr = base_model(
                lr_b
            ).clamp(0, 1)

            da_sr = da_model(
                lr_b
            ).clamp(0, 1)

        bic_psnr_list.append(
            psnr(bic, hr_b).item()
        )

        bic_ssim_list.append(
            ssim(bic.float(), hr_b.float()).item()
        )

        base_psnr_list.append(
            psnr(base_sr, hr_b).item()
        )

        base_ssim_list.append(
            ssim(base_sr.float(), hr_b.float()).item()
        )

        da_psnr_list.append(
            psnr(da_sr, hr_b).item()
        )

        da_ssim_list.append(
            ssim(da_sr.float(), hr_b.float()).item()
        )

    row = {
        "Condition": level,

        "Bicubic PSNR":
            np.mean(bic_psnr_list),

        "BASE PSNR":
            np.mean(base_psnr_list),

        "DA PSNR":
            np.mean(da_psnr_list),

        "DA Gain dB":
            np.mean(da_psnr_list)
            - np.mean(base_psnr_list),

        "Bicubic SSIM":
            np.mean(bic_ssim_list),

        "BASE SSIM":
            np.mean(base_ssim_list),

        "DA SSIM":
            np.mean(da_ssim_list),

        "DA SSIM Gain":
            np.mean(da_ssim_list)
            - np.mean(base_ssim_list)
    }

    results.append(row)


# ==========================================================
# 7. FINAL TABLE
# ==========================================================

df = pd.DataFrame(results)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.4f}"
)

print("\n========================================")
print("BASE vs DA DEGRADATION STRESS TEST")
print("========================================\n")

print(df.to_string(index=False))

df.to_csv(
    "/kaggle/working/base_vs_da_stress_test.csv",
    index=False
)

print(
    "\nSaved:"
    "/kaggle/working/base_vs_da_stress_test.csv"
)

BASE: /kaggle/input/models/muhammadhassan2/base-best/other/default/1/base_best.pth
DA: /kaggle/input/models/muhammadhassan2/da-best/other/default/1/da_best.pth
Device: cuda
Validation images: 420
BASE checkpoint epoch: 74
DA checkpoint epoch: 92

Testing: Bicubic

Testing: Mild

Testing: Medium

Testing: Severe

BASE vs DA DEGRADATION STRESS TEST

Condition  Bicubic PSNR  BASE PSNR  DA PSNR  DA Gain dB  Bicubic SSIM  BASE SSIM  DA SSIM  DA SSIM Gain
  Bicubic       25.1951    26.2208  26.2894      0.0686        0.6845     0.7245   0.7298        0.0054
     Mild       24.7020    25.6596  25.6974      0.0378        0.6546     0.6970   0.7008        0.0039
   Medium       23.8123    24.5080  24.5173      0.0093        0.6045     0.6399   0.6410        0.0012
   Severe       22.9019    23.3962  23.4029      0.0067        0.5595     0.5871   0.5877        0.0006

Saved:/kaggle/working/base_vs_da_stress_test.csv
